# Lakehouse Monitoring (v0.1): Create & Refresh Monitors from `monitors_control`

This notebook:
1) Reads enabled rows from **`<catalog>.<admin_schema>.monitors_control`**  
2) Creates or updates Lakehouse Monitors (no custom metrics in v0.1)  
3) Triggers a refresh so profile tables are populated

> v0.1 scope: **only** `monitors_control` (no `metric_templates` / `metric_bindings`).

In [0]:
# Widgets to locate the admin table
dbutils.widgets.text("catalog",      "dbdemos_steventan",  "Catalog")
dbutils.widgets.text("admin_schema", "monitoring_admin",   "Admin schema")

catalog      = dbutils.widgets.get("catalog").strip()
admin_schema = dbutils.widgets.get("admin_schema").strip()

ADMIN = f"{catalog}.{admin_schema}"
print(f"Using admin schema: {ADMIN}")

In [0]:
from pyspark.sql import functions as F

ctrl_path = f"{ADMIN}.monitors_control"
ctrl_df = (spark.table(ctrl_path)
                .filter(F.col("enabled") == True)
                .select(
                    "table_catalog","table_schema","table_name",
                    "profile_type","timestamp_col","granularities",
                    "output_schema_name","assets_dir",
                    "schedule_cron","schedule_tz",
                    "notifications_on_failure","enable_cdf","enabled"
                ))

display(ctrl_df.orderBy("table_schema","table_name"))

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.catalog import (
    MonitorTimeSeries, MonitorCronSchedule, MonitorNotifications
)

w = WorkspaceClient()

def _mk_time_series(row):
    # time-series is optional, but recommended for profile_type='TimeSeries'
    ts_col = getattr(row, "timestamp_col", None)
    if not ts_col:
        return None
    grns = list(getattr(row, "granularities", None) or ["1 day"])
    return MonitorTimeSeries(timestamp_col=ts_col, granularities=grns)

def _mk_schedule(row):
    cron = getattr(row, "schedule_cron", None)
    tz   = getattr(row, "schedule_tz",   None)
    if cron and tz:
        return MonitorCronSchedule(quartz_cron_expression=cron, timezone_id=tz)
    return None

def _mk_notifications(row):
    emails = list(getattr(row, "notifications_on_failure", None) or [])
    if emails:
        return MonitorNotifications(on_failure={"email_addresses": emails})
    return None

In [0]:
# ────────────────────────────────────────────────
# Main loop (v0.1: only monitors_control, no custom metrics)
# ────────────────────────────────────────────────
results = []
for r in ctrl_df.collect():
    table_fqn = f"{r.table_catalog}.{r.table_schema}.{r.table_name}"

    desired = dict(
        table_name=table_fqn,
        output_schema_name=r.output_schema_name
    )
    if r.schedule_cron and r.schedule_tz:
        desired["schedule"] = MonitorCronSchedule(
            quartz_cron_expression=r.schedule_cron, timezone_id=r.schedule_tz
        )
    if r.notifications_on_failure:
        desired["notifications"] = MonitorNotifications(
            on_failure={"email_addresses": list(r.notifications_on_failure)}
        )
    if r.timestamp_col:
        desired["time_series"] = MonitorTimeSeries(
            timestamp_col=r.timestamp_col,
            granularities=list(r.granularities) if r.granularities else ["1 day"]
        )

    try:
        # If exists → update and refresh
        existing = w.quality_monitors.get(table_name=table_fqn)
        w.quality_monitors.update(**desired)
        w.quality_monitors.run_refresh(table_name=table_fqn)
        action, note = "updated", "refresh queued"
    except Exception:
        # If not exists → create, but DO NOT refresh (let system run first profile)
        w.quality_monitors.create(assets_dir=r.assets_dir, **desired)
        action, note = "created", "first scheduled run will populate"

    results.append((table_fqn, action, note))

for t,a,n in results:
    print(f"• {t} -> {a} | {n}")

In [0]:
from time import sleep

def monitor_status(table_fqn: str):
    try:
        m = w.quality_monitors.get(table_name=table_fqn)
        # Status enum names vary slightly by SDK versions; normalize to string
        st = getattr(m, "status", None) or getattr(m, "monitor_status", None)
        return str(st)
    except Exception as e:
        return f"ERROR: {e.__class__.__name__}"

print("Checking statuses for enabled controls...\n")
for r in ctrl_df.select("table_catalog","table_schema","table_name").collect():
    t = f"{r.table_catalog}.{r.table_schema}.{r.table_name}"
    print(f"[{t}] status = {monitor_status(t)}")

In [0]:
print("Profile output tables are created per entry in monitors_control:")
display(
  ctrl_df.select(
      "table_schema","table_name","output_schema_name"
  ).withColumn(
      "profile_table_fqn", F.concat_ws(".", F.col("output_schema_name"), F.concat(F.col("table_name"), F.lit("_profile_metrics")))
  )
)

## Notes & next steps

- v0.1 uses only **`monitors_control`** to create/update monitors and refresh them.
- Results are written to `<output_schema_name>.<table>_profile_metrics`.
- For v0.2 you can:
  - Add `metric_templates` and `metric_bindings` to define custom rules
  - Build dynamic views for dashboards (ratio + details pairing)
  - Add idempotent seeding notebooks to produce realistic data drift